# FQML-NIDS · LEARN-1 · The ideas behind E0, on our own data

**For:** Ganna, Kareem, Sheghaf (and anyone reviewing E0) · **By:** Nouran (AI and quantum ML) · **Time:** about 45 minutes

Phase 1 asks one question: *on CICIoT2023, tested without leakage, does our small quantum model beat classical models of the same size?*
Every rule in `docs/PREREG_E0.md` and `docs/DECISIONS.md` exists because of one of the six ideas below. Each section shows the idea on
**real CICIoT2023 rows** (a small sample of our cleaned file), so you see the effect rather than take it on trust.

1. Train, validation, test: three jobs, three sets
2. Overfitting
3. Why F1 hid a 33–52% false-alarm rate
4. Thresholds at a fixed alarm rate
5. Leakage: why the random split inflated scores, and the `Number` shortcut
6. What a VQC is, and why RFF is its fair rival

At the end: **5 check questions** (answers hidden). LEARN-1 is done when you have answered them.

**Before running:** the shared folder `FQML-NIDS-data` must be reachable from your Drive (*Add shortcut to Drive* if it was shared with you). Runtime: CPU is enough.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, pathlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split

FEATURES = ['Header_Length', 'Protocol Type', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number',
            'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count',
            'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP',
            'ARP', 'ICMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'Variance']
FAMILIES = ['Benign', 'DDoS', 'DoS', 'Recon', 'Web', 'BruteForce', 'Spoofing', 'Mirai']
plt.rcParams['figure.figsize'] = (7, 3.5)

## 0. A small, ordered sample of our data
The cleaned file has 46.8 M rows. We take, for each of the 34 traffic types, up to 3 of its capture files and 1,500 rows spread evenly
over each, **keeping the recorded order** (`row_id`). That is about 100,000 rows: enough to see every effect in a minute or two.
The 35 features are the model inputs after D9 (no `IAT`, no `Number`); `Number` is loaded separately for section 5.

In [ ]:
def find_parquet():
    pats = ['/content/drive/MyDrive/**/ciciot2023_clean_v1.parquet', '/content/drive/Shareddrives/**/ciciot2023_clean_v1.parquet',
            '/content/drive/MyDrive/.shortcut-targets-by-id/**/ciciot2023_clean_v1.parquet']
    for p in pats:
        hits = glob.glob(p, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError('ciciot2023_clean_v1.parquet not found: add a shortcut to FQML-NIDS-data in My Drive')


def load_sample(path, rows_per_file=1500, files_per_label=3):
    """A small, ordered sample: for every attack type, up to 3 of its capture files (first, middle, last),
    1,500 rows spread evenly over each file. Row order (row_id) = recorded order inside each file."""
    import pyarrow.parquet as pq
    pf = pq.ParquetFile(path)
    names = pf.schema_arrow.names
    j = names.index('native_label')
    label_of = []
    for i in range(pf.num_row_groups):               # one row group per capture file (cleaner v1)
        st = pf.metadata.row_group(i).column(j).statistics
        label_of.append(st.min if (st is not None and st.has_min_max and st.min == st.max)
                        else pf.read_row_group(i, columns=['native_label']).column(0)[0].as_py())
    groups = {}
    for i, lab in enumerate(label_of):
        groups.setdefault(lab, []).append(i)
    cols = ['row_id', 'native_label', 'family', 'dup_group', 'Number', *FEATURES]
    parts = []
    for lab, gs in groups.items():
        pick = sorted(set([gs[0], gs[len(gs) // 2], gs[-1]]))[:files_per_label]
        for g in pick:
            t = pf.read_row_group(g, columns=cols)
            k = np.unique(np.linspace(0, t.num_rows - 1, min(rows_per_file, t.num_rows)).round().astype(int))
            parts.append(t.take(k).to_pandas())
    df = pd.concat(parts, ignore_index=True).sort_values('row_id', ignore_index=True)
    df['y'] = df['family'].map({f: i for i, f in enumerate(FAMILIES)}).astype(int)
    return df


PATH = find_parquet()
t0 = time.time()
df = load_sample(PATH)
print(f'{len(df):,} rows in {time.time()-t0:.0f} s')
df.groupby('family').size().rename('rows').to_frame().T

## 1. Train, validation, test: three jobs, three sets
- **Train**: the rows the model learns from.
- **Validation**: rows the model never learns from, used to *choose* things: when to stop training, which of the 12 tuning trials wins,
  where to put the alarm threshold.
- **Test**: rows used **once**, at the end, to report the score. If you look at test rows while choosing anything, the reported score
  is no longer an honest estimate: you have tuned to the test set.

Two ways to make the sets:
- **Random rows**: shuffle and cut. Neighbouring windows of the same recording land on both sides.
- **Time-blocked** (our protocol A, simplified): inside every traffic type, the first 60% in recorded order trains, the next 20% validates,
  the last 20% tests, with a small gap at each boundary. The model is tested on *later* traffic than it learned from, as a deployed IDS would be.

In [ ]:
def random_split(df, seed=0):
    tr, rest = train_test_split(df.index, test_size=0.4, random_state=seed, stratify=df['native_label'])
    va, te = train_test_split(rest, test_size=0.5, random_state=seed, stratify=df.loc[rest, 'native_label'])
    return np.sort(tr), np.sort(va), np.sort(te)


def time_split(df, gap=20):
    """Inside every attack type, in recorded order: first 60% train, next 20% validation, last 20% test,
    `gap` rows dropped at each boundary (like protocol A, with one block each)."""
    tr, va, te = [], [], []
    for _, g in df.groupby('native_label', sort=False):
        idx = g.sort_values('row_id').index.to_numpy()
        n = len(idx)
        a, b = int(0.6 * n), int(0.8 * n)
        tr.append(idx[:max(a - gap, 0)]); va.append(idx[a:max(b - gap, a)]); te.append(idx[b:])
    return tuple(np.sort(np.concatenate(x)) for x in (tr, va, te))


def macro_f1(y, p):
    return f1_score(y, p, average='macro', labels=list(range(8)), zero_division=0)


for name, s in (('random rows', random_split(df)), ('time-blocked', time_split(df))):
    print(f'{name:13s} train {len(s[0]):,}  validation {len(s[1]):,}  test {len(s[2]):,}')

## 2. Overfitting
A decision tree can keep splitting until it memorises its training rows. As it gets deeper, the **training** score keeps rising; the
**validation** score rises, then flattens or falls. The gap between the two lines is overfitting. This is why every E0 model is stopped
early on validation (patience 10) and keeps its *best validation* weights, not its last ones (H-10).

In [ ]:
def overfit_curve(df, split, depths=range(1, 31, 2)):
    tr, va, _ = split
    X, y = df[FEATURES].to_numpy(), df['y'].to_numpy()
    rows = []
    for d in depths:
        m = DecisionTreeClassifier(max_depth=d, random_state=0, class_weight='balanced').fit(X[tr], y[tr])
        rows.append((d, macro_f1(y[tr], m.predict(X[tr])), macro_f1(y[va], m.predict(X[va]))))
    return pd.DataFrame(rows, columns=['depth', 'train', 'validation'])


curve = overfit_curve(df, time_split(df))
ax = curve.plot(x='depth', y=['train', 'validation'], marker='o', title='Decision tree, 8-class macro-F1')
ax.set_ylabel('macro-F1'); plt.show()
curve.round(3).T

## 3. Why F1 hid a 33–52% false-alarm rate (Checkpoint 1, D11 part 1)
Binary task: attack or benign. Two numbers matter to an operator:
- **Recall** (detection rate): share of attacks caught.
- **FPR** (false-alarm rate): share of *benign* traffic flagged. An IDS that flags 30% of normal traffic is unusable, whatever its F1.

**F1** mixes precision and recall on the attack class. When attacks are 97% of the rows, a model can flag almost everything and still score
F1 ≈ 0.99: the few benign rows it wrongly flags barely move precision. Look at the first row of the table below: high F1, and see the FPR.

In [ ]:
def binary_view(df, split):
    tr, va, te = split
    X, attack = df[FEATURES].to_numpy(), (df['y'] != 0).to_numpy().astype(int)
    m = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(X[tr], attack[tr])
    s_va, s_te = m.predict_proba(X[va])[:, 1], m.predict_proba(X[te])[:, 1]
    p_te = (s_te >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(attack[te], p_te, labels=[0, 1]).ravel()
    default = {'F1': f1_score(attack[te], p_te), 'FPR': fp / max(fp + tn, 1), 'recall': tp / max(tp + fn, 1),
               'benign test rows': int(tn + fp), 'attack test rows': int(tp + fn)}
    out = {'default threshold 0.5': default}
    for target in (0.01, 0.001):
        ben = np.sort(s_va[attack[va] == 0])
        thr = ben[min(int(np.ceil((1 - target) * len(ben))), len(ben) - 1)] if len(ben) else 0.5
        p = (s_te > thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(attack[te], p, labels=[0, 1]).ravel()
        out[f'threshold for {target:.1%} FPR (set on validation)'] = {
            'F1': f1_score(attack[te], p), 'FPR': fp / max(fp + tn, 1), 'recall': tp / max(tp + fn, 1),
            'threshold': float(thr)}
    return pd.DataFrame(out).T, (attack[te], s_te)


table, (att_te, s_te) = binary_view(df, time_split(df))
table.round(4)

## 4. Thresholds at a fixed alarm rate
Instead of the default 0.5, fix the alarm rate you can afford **first**, then measure recall:
1. On the **validation** benign rows, find the score threshold that flags exactly 1% (or 0.1%) of them.
2. Apply that threshold, unchanged, to the **test** rows. Report recall and the actual test FPR.

That is "attack recall at 1% FPR", the binary measure of E0. The threshold comes from validation, never from test, for the reason in section 1.
The ROC curve below shows every threshold at once; ROC-AUC is the area under it.

In [ ]:
fpr, tpr, _ = roc_curve(att_te, s_te)
plt.plot(fpr, tpr); plt.xscale('log'); plt.xlim(1e-4, 1)
for t in (0.01, 0.001): plt.axvline(t, ls='--', c='grey')
plt.xlabel('false-alarm rate (log scale)'); plt.ylabel('attacks caught'); plt.title('ROC on test rows'); plt.show()

## 5. Leakage
**Leakage** = information from the test rows reaches training, so the test score is better than real-world performance.

**(a) Neighbours and twins.** Consecutive windows of one recording are near-copies; 61% of CICIoT2023 rows even have an *exact* copy (D8,
DATA-4). With random rows, a test row's twin is often in training, and the model is graded partly on memory. Compare the twin share and the
score of the two splits below. (On the full data, Checkpoint 1 measured 0.628 → 0.575 8-class macro-F1 from random to protocol A.)

**(b) Shortcuts.** CICIoT2023 averages each feature over a window of **100 packets** for DDoS, DoS and Mirai and **10 packets** for the rest.
The column `Number` *is* that window size. Alone, it separates the two groups perfectly, but it describes how CIC *recorded* each attack, not
the traffic. A deployed IDS uses one window size, so a model leaning on `Number` would fail there. That is why `Number` (and `IAT`) are dropped
(D9). XGBoost had put 40–47% of its importance on it.

In [ ]:
def leakage_table(df):
    X, y = df[FEATURES].to_numpy(), df['y'].to_numpy()
    rows = []
    for name, split in (('random rows', random_split(df)), ('time-blocked', time_split(df))):
        tr, va, te = split
        m = HistGradientBoostingClassifier(max_iter=200, random_state=0, class_weight='balanced').fit(X[tr], y[tr])
        twins = np.isin(df['dup_group'].to_numpy()[te], df['dup_group'].to_numpy()[tr]).mean()
        rows.append({'split': name, 'test 8-class macro-F1': macro_f1(y[te], m.predict(X[te])),
                     'test rows with an exact twin in training': twins})
    return pd.DataFrame(rows)


def shortcut_table(df):
    """One feature alone: Number (the packet-window size) separates 100-packet families from the rest."""
    split = time_split(df)
    tr, _, te = split
    big = df['family'].isin(['DDoS', 'DoS', 'Mirai']).to_numpy().astype(int)
    rows = []
    for f in ['Number', 'Rate', 'Tot sum']:
        X = df[[f]].to_numpy()
        m = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X[tr], big[tr])
        acc = (m.predict(X[te]) == big[te]).mean()
        rows.append({'feature alone': f, 'accuracy: 100-packet family or not': acc,
                     'values of Number seen': ', '.join(map(str, sorted(df[f].round(1).unique())[:5])) if f == 'Number' else ''})
    return pd.DataFrame(rows)


leakage_table(df).round(3)

In [ ]:
shortcut_table(df).round(3)

## 6. What a VQC is, and why RFF is its fair rival
Our VQC has **8 qubits, one per feature**. Each layer:
1. **Encode**: rotate qubit q by the feature value, `RY(x_q)` (inputs scaled to [0, π], DATA-6).
2. **Trainable rotations** `RZ(θ)`, `RY(θ)` on every qubit: 2 angles × 8 qubits × 3 layers = 48 trainable angles.
3. **CNOT chain**: entangling gates that let qubits influence each other (`vqc_noent` removes them).

At the end each output is `⟨Z⟩` (between −1 and +1) times a trained weight, plus a bias: 64 parameters in all for 8 classes.

**The key fact.** Because x enters through rotations, the output is always a sum of **sines and cosines of x**, and the highest frequency is the
number of times x is encoded. Below, a 2-qubit toy that encodes the same x on both qubits (twice per layer): its output contains frequencies up
to 2, 4, 6 for 1, 2, 3 layers and nothing above.

So a classical model that builds those sines and cosines directly (**random Fourier features, RFF**) and fits a linear layer on top can represent
the same kind of function. If our VQC does not beat RFF, the quantum circuit adds nothing a cheap classical model cannot do. That is
comparisons 3–4 of PREREG.

In [ ]:
I2 = np.eye(2)
Z = np.diag([1.0, -1.0])


def ry(a):
    c, s = np.cos(a / 2), np.sin(a / 2)
    return np.array([[c, -s], [s, c]])


def rz(a):
    return np.diag([np.exp(-0.5j * a), np.exp(0.5j * a)])


CNOT = np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=complex)


def tiny_vqc(x, theta):
    """2 qubits, both get the same input x (angle encoding RY(x)); each layer: encode, RZ, RY, then CNOT.
    theta: [layers, 2, 2]. Returns <Z> on qubit 1."""
    psi = np.zeros(4, dtype=complex); psi[0] = 1
    for layer in theta:
        u = [ry(layer[q, 0]) @ rz(layer[q, 1]) @ ry(x) for q in range(2)]
        psi = CNOT @ (np.kron(u[0], u[1]) @ psi)
    return float(np.real(np.conj(psi) @ np.kron(I2, Z) @ psi))


def spectrum(layers, seed=0, n=256):
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, (layers, 2, 2))
    xs = np.linspace(0, 2 * np.pi, n, endpoint=False)
    f = np.array([tiny_vqc(x, theta) for x in xs])
    c = np.abs(np.fft.rfft(f)) / n
    return xs, f, c

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for L in (1, 2, 3):
    xs, f, c = spectrum(L)
    axes[0].plot(xs, f, label=f'{L} layer(s)')
    axes[1].bar(np.arange(9) + (L - 2) * 0.25, c[:9], width=0.25, label=f'{L} layer(s)')
axes[0].set_xlabel('x'); axes[0].set_ylabel('<Z>'); axes[0].legend(); axes[0].set_title('Output as x varies')
axes[1].set_xlabel('frequency'); axes[1].set_ylabel('|Fourier coefficient|'); axes[1].legend(); axes[1].set_title('Only low frequencies appear')
plt.tight_layout(); plt.show()

## Check questions
Answer in the team chat (one line each is enough), then open the answers.

1. Why must the alarm threshold be chosen on validation rows and not on test rows?
2. A model has binary F1 0.995. Why is that not enough to call it a good IDS, and what two numbers would you ask for?
3. In section 5, why does the random split give a higher score than the time-blocked one, even though it is the same model?
4. `Number` separates DDoS/DoS/Mirai from the rest perfectly. Why do we still drop it?
5. Why is RFF, not only an MLP of the same size, the right rival to decide whether the *quantum* part of the VQC helps?

<details><summary><b>Answers</b> (open after answering)</summary>

1. Anything chosen by looking at test rows is tuned to them, so the test score stops being an honest estimate of new traffic. Validation exists to make choices; test is used once.
2. F1 can stay near 1 while a large share of benign traffic is flagged, because attacks dominate the rows. Ask for the false-alarm rate and for attack recall at a fixed false-alarm rate (1% and 0.1%).
3. Random rows put near-copies and exact twins of test rows into training, so part of the test score is memory. Time-blocked tests on later traffic of each type, as deployment would.
4. It is the recording window size (100 vs 10 packets), a property of how the dataset was made, not of the traffic; a deployed IDS uses one window size, so the shortcut would not transfer.
5. The VQC's output is a sum of sines and cosines of its inputs with bounded frequencies; RFF builds exactly that function class classically. Beating an MLP could come from that function class alone; beating RFF is evidence the circuit itself adds something.
</details>